# 05 - Đánh giá Toàn diện 3-Fold CV và Tạo File Submission

Trong bài học kết thúc này:
- Nạp session chung `lesson_session`, hỗ trợ lựa chọn phương pháp: `METHOD = 'lr' | 'center60' | 'native' | 'b2' | 'blend'`.
- Hoàn thiện huấn luyện các fold còn thiếu và đánh giá Out-Of-Fold (OOF) trên toàn bộ 800 cặp development.
- Phân tích lưu ý về hiện tượng Optimistic Bias khi chọn checkpoint bằng tập validation.
- Quy trình Inference an toàn trên tập Test không nhãn (hoặc báo `no submission produced` nếu chưa có tập test).
- Xuất file `submission.csv` chuẩn định dạng.

In [ ]:
from pathlib import Path
import os
import sys

def find_task_root():
    cwd = Path.cwd().resolve()
    for cand in [cwd, cwd.parent, cwd / 'KeMaoDanh', cwd.parent / 'KeMaoDanh']:
        if (cand / 'src/kmd').is_dir() and (cand / 'configs').is_dir():
            return cand.resolve()
    raise FileNotFoundError("Mở notebook từ repo root, KeMaoDanh hoặc KeMaoDanh/notebooks.")

TASK_ROOT = find_task_root()
if str(TASK_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(TASK_ROOT / 'src'))

import numpy as np
import pandas as pd

from kmd.core import PACKAGE, read_csv, read_json, metric, export_submission
from kmd.pipeline import (
    prepare_development, load_session, complete_method_cv,
    load_pairs, check_test_separation, infer_cnn, infer_lr, blend, aligned_predictions
)

RUN_ID = 'lesson_session'
# Lựa chọn phương pháp: 'lr', 'center60', 'native', 'b2', hoặc 'blend'
METHOD = 'lr'

print(f"Phiên làm việc: {RUN_ID} | Phương pháp lựa chọn: {METHOD}")

## 1. Mở rộng 3-Fold Cross-Validation (Full 800 OOF)

Hàm `complete_method_cv` sẽ:
- Tự động nhận diện và tái sử dụng an toàn các fold đã hoàn thành từ các bài trước (ví dụ Fold 0).
- Huấn luyện các fold còn thiếu (Fold 1, Fold 2).
- Lập bảng dự đoán Out-Of-Fold (OOF) đầy đủ cho 800 cặp development.

| Lượt | Train | Validation |
|---|---|---|
| 0 | Nhóm 1 + 2 | Nhóm 0 |
| 1 | Nhóm 0 + 2 | Nhóm 1 |
| 2 | Nhóm 0 + 1 | Nhóm 2 |

Mỗi cặp development nhận đúng một dự đoán từ mô hình không fit trên cặp đó. Ghép ba phần validation tạo bảng OOF 800 hàng; **không** lấy trung bình ba model trên development. Với test mới, cả ba model đều chưa fit trên test nên inference lấy trung bình ba xác suất.

METHOD='lr' chạy được ngay sau bài 01 và fit lại ba LR nhẹ trên CPU. Phương pháp CNN chỉ dùng lại fold hoàn chỉnh khớp hợp đồng dữ liệu/code/config; fold thiếu sẽ train, fold dở dang báo lỗi và yêu cầu RUN_ID mới. Giữ cùng RUN_ID trong các bài 01–05. Thay code, ảnh, split hoặc cấu hình là một thí nghiệm mới.

In [ ]:
data_root_env = os.environ.get('DATA_ROOT')
data_root = Path(data_root_env or TASK_ROOT / 'data/train').expanduser().resolve()
if not (data_root / 'pairs.csv').is_file():
    raise FileNotFoundError(f'Thiếu dữ liệu train: {data_root / "pairs.csv"}. Xem README để đặt DATA_ROOT.')

if (data_root / 'pairs.csv').is_file():
    dev_frame = prepare_development(data_root)
    session_dir = load_session(RUN_ID, data_root)
    print(f"Nạp session tại: {session_dir.name}")
    
    oof_predictions = complete_method_cv(METHOD, dev_frame, data_root, session_dir)
    score_oof = metric(dev_frame.fake_position, oof_predictions.p)
    
    print(f"\n=== KẾT QUẢ ĐÁNH GIÁ 3-FOLD FULL OOF ({METHOD.upper()}) ===")
    print(f"- Macro-F1: {score_oof['macro_f1']:.4f}")
    print(f"- Accuracy: {score_oof['accuracy']:.4f}")
    print(f"- Log Loss: {score_oof['log_loss']:.4f}")
    print(f"- Tổng số lỗi: {score_oof['errors']} / 800")
else:
    print("Cần dataset tại data/train để chạy đánh giá 3-fold.")

## 2. Lưu ý về việc chọn Checkpoint bằng Validation

Trong các nhánh CNN có Early Stopping, file trọng số `best.pt` được chọn tại epoch có validation metric tốt nhất. Do đó, điểm số OOF có xu hướng hơi lạc quan (optimistic bias) so với dữ liệu kiểm tra thực tế chưa từng thấy.

Với native/resampled chạy 19 epoch cố định, file best.pt lưu trạng thái cuối theo giao thức đó; tên file không đồng nghĩa tất cả nhánh đều chọn epoch tốt nhất. Dù vậy, chọn phương pháp sau khi so sánh nhiều lần trên development vẫn tạo thiên lệch lựa chọn.

## 3. Quy trình Inference trên Tập Test Không Nhãn

In [ ]:
test_root_env = os.environ.get('TEST_ROOT')
test_root = Path(test_root_env or TASK_ROOT / 'data/test').expanduser().resolve()

has_test = (test_root / 'pairs.csv').is_file()
if test_root_env and not has_test:
    raise FileNotFoundError(f'TEST_ROOT đã đặt nhưng thiếu pairs.csv: {test_root}')

if has_test and (data_root / 'pairs.csv').is_file():
    test_manifest = load_pairs(test_root, labeled=False)
    check_test_separation(test_manifest, test_root, session_dir)
    print(f"Đã nạp và kiểm tra tách biệt {len(test_manifest)} cặp ảnh test.")
    
    # Inference theo phương pháp đã chọn
    if METHOD == 'blend':
        pred_b2 = infer_cnn('b2', test_manifest, test_root, session_dir)
        pred_nat = infer_cnn('native', test_manifest, test_root, session_dir)
        test_pred = blend(pred_b2, pred_nat, test_manifest)
    elif METHOD == 'lr':
        test_pred = infer_lr(test_manifest, test_root, session_dir)
    else:
        test_pred = infer_cnn(METHOD, test_manifest, test_root, session_dir)
        
    # Căn chỉnh đảm bảo đúng thứ tự dòng với test pairs.csv
    test_pred = aligned_predictions(test_pred, test_manifest)
    assert test_pred.pair_id.tolist() == test_manifest.pair_id.tolist()
    
    out_dir = PACKAGE / 'outputs' / session_dir.name
    out_dir.mkdir(parents=True, exist_ok=True)
    
    test_pred.to_csv(out_dir / 'test_probabilities.csv', index=False)
    sub_df = export_submission(test_pred, out_dir / 'submission.csv')
    print(f"Đã xuất submission thành công tại: {out_dir / 'submission.csv'}")
    print(sub_df.head())
else:
    print("Thông báo: Chưa có tập dữ liệu test. Quá trình huấn luyện và đánh giá development hoàn tất an toàn.")
    print("Trạng thái: no submission produced (test root not provided).")

## 4. Tổng kết toàn bộ chuỗi bài giảng

Chuỗi bài giảng đã đi qua đầy đủ các công đoạn từ xử lý dữ liệu, trích xuất đặc trưng thủ công, fine-tuning CNN, multi-crop native, phân tích lỗi, blending và xuất file nộp bài.

Để chạy lại toàn bộ pipeline trong một lượt, bạn có thể sử dụng [pipeline_end_to_end.ipynb](pipeline_end_to_end.ipynb).